<a href="https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HashamHassan-01/flyrank-ml-internship-hasham/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!pip -q install duckdb huggingface_hub

In [ ]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [ ]:
from google.colab import userdata


HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded successfully!" if HF_TOKEN else "Token not found!")

Token loaded successfully!


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [ ]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


In [ ]:
!pip -q install duckdb pandas pyarrow huggingface_hub

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents the daily search performance of one content item (content_hash_id) for one client (client_hash_id) on one report date (report_date).

### Time Window

For this assignment, I use the March 2026 partition (2026-03-01 to 2026-03-31). This is a mid-panel month recommended by FlyRank, helping avoid using the final month as the outcome window and reducing the risk of data leakage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields: Feature / Label / Context / Excluded

### Features
These fields represent information available at prediction time from the current observation window and can be used as model inputs.

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_users

### Label / Proxy
The target will be whether a content item's search performance declines in a future period. This label will be created later and is **not used as a feature**.

### Context
These fields identify records but are not used for prediction.

- client_hash_id
- content_hash_id
- report_date
- month

### Excluded

Future performance metrics — excluded because they would cause label leakage.

Identifiers such as client_hash_id and content_hash_id are excluded from modeling because they identify entities but do not represent predictive signals.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 5
""").df()



,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



## 3. Verify it with queries (grain, counts, missing values, windows)

The following checks verify the data contract assumptions:

1. Confirm that each row represents one content item for one client on one report date.
2. Verify the March 2026 time window and total number of records.
3. Check missing values for important feature fields.
4. Verify availability of GA4 measurements.

In [ ]:
print("Query 1: Verify grain (content + client + date uniqueness)")

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(
        DISTINCT
        content_hash_id || '-' || client_hash_id || '-' || CAST(report_date AS VARCHAR)
    ) AS unique_content_client_date_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()


print("Query 2: Verify March 2026 time window and row count")

con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()


print("Query 3: Check missing values in feature columns")

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END)
        AS missing_gsc_impressions,

    SUM(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END)
        AS missing_gsc_clicks,

    SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END)
        AS missing_gsc_avg_position,

    SUM(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END)
        AS missing_ga4_sessions,

    SUM(CASE WHEN ga4_users IS NULL THEN 1 ELSE 0 END)
        AS missing_ga4_users

FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()


print("Query 4: Verify GA4 availability")

con.sql(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS row_count
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY ga4_data_available
ORDER BY row_count DESC
""").df()


print("Query 5: Confirm selected fields exist")

con.sql(f"""
DESCRIBE
SELECT *
FROM {TABLES['fact_daily']}
""").df()

Query 1: Verify grain (content + client + date uniqueness)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2: Verify March 2026 time window and row count
Query 3: Check missing values in feature columns


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 4: Verify GA4 availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 5: Confirm selected fields exist


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

- History length differs across clients, so not every client has the same amount of historical data.
- Some records have unavailable GA4 measurements (ga4_data_available), which limits the reliability of engagement-based features.
- This dataset cannot explain why search performance changed; it only measures observed search and analytics metrics.
- The analysis is intended for decision support and should not be interpreted as proving causation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    ga4_data_available,
    COUNT(*) AS rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY ga4_data_available
""").df()

,ga4_data_available,rows
0,<NA>,3018741
1,False,6408671
2,True,413966


## Self-check

## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.